# Pydantic AI and Nvidia Build models
PydanticAI is a framework to create applications with AI models. PydanticAI is model agnostic and can use several models. In this lecture we will use Nvidia Builds models as they provide good quota limits (40 requests per minute as of this writing).

In [1]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

It imports the Agent class from the pydantic_ai library, which you installed with uv add. After that line, you can write Agent(...) in your code.

An import doesn’t run anything or contact any model. It only makes the name Agent available, like taking a tool out of a toolbox.

Agent is the main object in Pydantic AI. You give it a model (your Nvidia one) and it handles sending your prompts and returning the answers. In your code, agent.run("...") is what sends the question.

Your original code used Agent(model) without this import, so Python had no idea what Agent was. That would have raised NameError: name 'Agent' is not defined.

In [2]:
# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat interface
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # model name; any model from build.nvidia.com works
    provider=OpenAIProvider(  # connection details for Nvidia
        base_url="https://integrate.api.nvidia.com/v1",  # Nvidia's API address
        api_key=os.environ["NVIDIA_API"],  # your secret key, read from the environment
    ),
)

# The agent wraps the model. Later you can give it instructions, tools, etc.
agent = Agent(model)

# Send a question to the model and wait for the answer.
# `await` pauses until the model replies. It works directly in a Jupyter notebook;
# in a normal .py script you would need asyncio.run(...) instead (see below).
result = await agent.run("What is the capital of Sweden?", timeout=60)

# result.output holds the model's text answer
print(result.output)

                 pydantic-ai v2.62.0 • Python 3.13.7

      / \        agent: agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


The capital of Sweden is **Stockholm**.


In [5]:
result = await agent.run("Tell a joke", timeout = 120)
result

AgentRunResult(output="Why don't scientists trust atoms? Because they make up everything!")

In [6]:
result.output

"Why don't scientists trust atoms? Because they make up everything!"

# Update the model on the fly

In [7]:
new_model = OpenAIChatModel(
    "nvidia/nemotron-3.5-bolt-e4b",  # any model from build.nvidia.com
    provider=OpenAIProvider( # connection details to nvidia
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NVIDIA_API"],  
    ),
)

result = await agent.run("Tell me a python joke", model=new_model)
result

AgentRunResult(output="Why did the Python programmer quit his job?\n\n**Because he didn't get enough *indents*!** 🐍😂")

### This model just HALLUCINATED

In [9]:
result = await agent.run(
    "Explain how Pydanticai differs from other AI frameworks?",
    model_settings = {"timeout":60},
)
result.output

'I want to make sure I\'m answering the right question: **"Pydanticai"** isn\'t a widely recognized name for a major AI framework in the current literature or industry. It\'s possible you\'re referring to one of the following:\n\n- **Pydantic** (the Python data validation/library schema library), often used with LLMs to enforce structured outputs\n- **Pydantic-AI** or related integrations that wrap Pydantic for LLM output parsing\n- A typo for **PyTorch**, **PaddlePaddle**, or another "-torch"/"-addle" framework\n- A very new or niche project not yet in broad circulation\n\nBelow I\'ll cover the most likely intent—**Pydantic-based AI integration**—and then contrast it with the major categories of AI frameworks, so you can identify which comparison you actually need.\n\n### If you meant Pydantic (or Pydantic-AI) in an AI context:\nPydantic isn\'t a "framework" for building or training models; it\'s a **data validation and parsing library** that uses Python type annotations. When paired 

In [ ]:
with open("pydanticai.md", "w") as file:
    file.write(result.output)

UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 3277: character maps to <undefined>